# 06. 한국어 텍스트 → 음성(TTS)

**실습 목표**  
한국어 문장을 Hugging Face의 사전학습 TTS 모델로 음성 파형으로 변환한다.

**주요 Hugging Face 모델**  
`facebook/mms-tts-kor`

> 이 노트북은 **파인튜닝 없이 사전학습 모델을 추론에 활용**하는 실습이다.  
> RTX 4060 8GB 환경을 고려했으며, CUDA가 없으면 CPU로 자동 전환하도록 구성하였다.

In [1]:
# uv add transformers accelerate scipy

In [2]:
import torch
print("PyTorch:", torch.__version__)

# 컴퓨터에 그래픽카드(GPU)가 있으면 계산이 훨씬 빨라져요. GPU가 있는지 확인!
print("CUDA available:", torch.cuda.is_available())

# transformers의 pipeline 기능은 GPU 번호를 숫자로 받아요 (0번 GPU, 없으면 -1 = CPU 사용)
DEVICE = 0 if torch.cuda.is_available() else -1
# torch 자체는 "cuda"(GPU) 또는 "cpu"라는 글자로 표시해요. 표기 방식만 다를 뿐 같은 의미예요
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", TORCH_DEVICE)

PyTorch: 2.14.0+cu126
CUDA available: True
device: cuda


In [ ]:
# sudo apt update

# sudo apt install -y build-essential

# sudo apt install -y python3.12-dev


In [5]:
from transformers import pipeline
from IPython.display import Audio  # 노트북에서 소리를 재생해주는 도구
import torch

DEVICE = 0 if torch.cuda.is_available() else -1

# 글자를 입력하면 사람 목소리처럼 들리는 음성을 만들어주는 "TTS(Text-to-Speech)" 파이프라인이에요
tts = pipeline(
    "text-to-speech",
    model="nineninesix/kani-tts-400m-ko",  # 한국어 음성을 생성하는 모델
    device=DEVICE
)

# 음성으로 바꿀 한국어 문장이에요
text = "안녕하세요. 허깅페이스 멀티모달 인공지능 실습을 시작합니다."

# 모델이 문장을 읽고 실제 소리(음성 파형)를 만들어내요
speech = tts(text)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [6]:
# speech 안에 어떤 정보(키)가 들어있는지 다시 한번 확인
print(speech.keys())

dict_keys(['audio', 'sampling_rate'])


In [7]:
print(speech.keys())
# 1초에 소리를 몇 번 재는지(샘플링 레이트)를 확인해요. 값이 없으면 기본값을 따로 정해줘야 해요
print("sampling_rate:", speech.get("sampling_rate"))
# 음성 데이터가 어떤 자료형인지 확인 (숫자 배열 형태예요)
print("audio type:", type(speech["audio"]))
# 음성 데이터의 크기(길이)를 확인해요
print("audio shape:", speech["audio"].shape)


dict_keys(['audio', 'sampling_rate'])
sampling_rate: None
audio type: <class 'numpy.ndarray'>
audio shape: (283,)


In [8]:
# tts 모델이 내부적으로 어떤 설정값(구조, 크기 등)을 쓰는지 자세히 확인해봐요
print(tts.model.config)

Lfm2Config {
  "_from_model_config": true,
  "architectures": [
    "Lfm2ForCausalLM"
  ],
  "block_auto_adjust_ff_dim": true,
  "block_dim": 1024,
  "block_ffn_dim_multiplier": 1.0,
  "block_mlp_init_scale": 1.0,
  "block_multiple_of": 256,
  "block_norm_eps": 1e-05,
  "block_out_init_scale": 1.0,
  "block_use_swiglu": true,
  "block_use_xavier_init": true,
  "bos_token_id": 1,
  "conv_L_cache": 3,
  "conv_bias": false,
  "conv_dim": 1024,
  "conv_dim_out": 1024,
  "conv_use_xavier_init": true,
  "dtype": "bfloat16",
  "eos_token_id": 7,
  "full_attn_idxs": null,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 6656,
  "layer_types": [
    "conv",
    "conv",
    "full_attention",
    "conv",
    "conv",
    "full_attention",
    "conv",
    "conv",
    "full_attention",
    "conv",
    "full_attention",
    "conv",
    "full_attention",
    "conv",
    "full_attention",
    "conv"
  ],
  "max_position_embeddings": 128000,
  "model_type": "lfm2",
  "norm_eps"

In [9]:
# 만들어진 음성을 노트북에서 바로 재생할 수 있는 플레이어로 보여줘요 (rate는 재생 속도 기준이 되는 샘플링 레이트)
Audio(
    speech["audio"],
    rate=22000
)

In [10]:
import scipy.io.wavfile as wavfile  # 소리 데이터를 wav 파일로 저장해주는 도구
import numpy as np
import os

# output 폴더가 없으면 새로 만들어요 (이미 있으면 그냥 넘어가요)
os.makedirs("output", exist_ok=True)

# 음성 데이터를 numpy 배열로 만들고, 불필요한 차원은 없애줘요
audio = np.asarray(speech["audio"]).squeeze()

# 실제 컴퓨터 파일로 저장해요 (output/korean_tts.wav 파일이 생겨요)
wavfile.write(
    "output/korean_tts.wav",
    22000,
    audio
)

# 

## 안정적 실행

In [ ]:
# uv remove transformers
# uv remove protobuf

# uv add kani-tts   <-- 직접실행해서 해야함.
# uv add "transformers==4.57.1"